# Raw training-data inspection

This notebook validates and combines downloaded PSV records into `data/physionet/combined.parquet` when needed, then restricts descriptive summaries and charts to source-A training records. Source-A label summaries are used only to reproduce the patient-level split (seed 42); held-out measurements and labels are not included in the audit. No model is trained and no imputation, scaling, clipping, or row removal is performed.

Use the project's `.venv` kernel after installing the package with `python -m pip install -e .`. Downloaded records, Parquet data, and patient-level plots belong under the Git-ignored `data/` directory; do not commit them. Structural or schema errors stop the scan. Extreme values are review candidates, not automatically errors.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from matplotlib import pyplot as plt

notebook_path = globals().get('__vsc_ipynb_file__')
current = Path(notebook_path).resolve().parent if notebook_path else Path.cwd().resolve()
project_root = next(
    (
        path for path in (current, *current.parents)
        if (path / 'pyproject.toml').is_file()
        and (path / 'src' / 'sepsis_warning').is_dir()
    ),
    None,
)
if project_root is None:
    raise RuntimeError('Open this notebook from the SUTD-PML-team2-main project')

# Make the checkout's src-layout package available in the notebook kernel.
source_dir = str(project_root / 'src')
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
from sepsis_warning.data import DYNAMIC, STATIC, Patient, split_patients
from sepsis_warning.prepare import prepare_parquet

data_root = project_root / 'data'
data_dir = data_root / 'physionet'
data_file = data_dir / 'combined.parquet'
if not data_file.is_file():
    raw_records = sum(
        len(list((data_dir / f'training_set{hospital}').glob('*.psv')))
        for hospital in ('A', 'B')
    )
    if not raw_records:
        raise FileNotFoundError(
            f'No downloaded PSV records found under {data_dir}. '
            'Download records first with: '
            'python scripts/download_data.py --limit 500 --output data/physionet'
        )
    print('Combined Parquet is missing; validating and combining downloaded records...')
    preparation_summary = prepare_parquet(data_dir, data_file)
    print(
        f"Combined {preparation_summary['records']:,} records and "
        f"{preparation_summary['hourly_rows']:,} hourly rows."
    )

dataset = ds.dataset(data_file, format='parquet')
required_columns = {'patient_id', 'hospital', 'SepsisLabel', *DYNAMIC, *STATIC}
missing_columns = required_columns - set(dataset.schema.names)
if missing_columns:
    raise ValueError(f'Combined Parquet is missing required columns: {sorted(missing_columns)}')

# Read source-A labels only to reproduce the same stratified patient splits.
source_a_labels = dataset.to_table(
    columns=['patient_id', 'hospital', 'SepsisLabel'],
    filter=ds.field('hospital') == 'A',
)
source_a_summary = source_a_labels.group_by(
    ['hospital', 'patient_id']
).aggregate([
    ('SepsisLabel', 'max'),
    ('SepsisLabel', 'min'),
])
source_patients = [
    Patient(
        patient_id=f"{row['hospital']}:{row['patient_id']}",
        hospital=row['hospital'],
        frame=pd.DataFrame({'SepsisLabel': [int(row['SepsisLabel_max'])]}),
    )
    for row in sorted(source_a_summary.to_pylist(), key=lambda item: item['patient_id'])
]
# split_patients requires both sources; this placeholder is never used for selection.
source_patients.append(
    Patient('B:external-placeholder', 'B', pd.DataFrame({'SepsisLabel': [0]}))
)
splits = split_patients(source_patients, seed=42)
training_records = splits['train']
training_keys = {
    (record.hospital, record.patient_id.split(':', 1)[1])
    for record in training_records
}
if not training_keys:
    raise ValueError('No source-A training records available')
training_ids = sorted(patient_id for _, patient_id in training_keys)
# Prefer a septic stay that was not already positive at its first recorded hour.
eligible_examples = [
    (row['hospital'], row['patient_id'])
    for row in source_a_summary.to_pylist()
    if (row['hospital'], row['patient_id']) in training_keys
    and row['SepsisLabel_max'] == 1
    and row['SepsisLabel_min'] == 0
]
example_key = min(eligible_examples) if eligible_examples else min(training_keys)
plot_columns = ('HR', 'MAP', 'Temp', 'Resp')
feature_columns = [*DYNAMIC, *STATIC]
observed_values = {column: [] for column in plot_columns}
stats = {
    column: {'missing': 0, 'observed': 0, 'minimum': None, 'maximum': None}
    for column in feature_columns
}
record_lengths = {key: 0 for key in training_keys}
trajectory = []
hour_count = positive_hours = 0

# The scanner predicate excludes validation, calibration, internal-test and B rows.
scanner = dataset.scanner(
    columns=['patient_id', 'hospital', *feature_columns, 'SepsisLabel'],
    filter=(ds.field('hospital') == 'A') & ds.field('patient_id').isin(training_ids),
    batch_size=65_536,
)
for batch in scanner.to_batches():
    hours = batch.to_pandas()
    hour_count += len(hours)
    positive_hours += int(hours['SepsisLabel'].sum())
    for patient_id, count in hours.groupby('patient_id', sort=False).size().items():
        key = ('A', patient_id)
        if key not in record_lengths:
            raise ValueError(f'Unexpected non-training record in scan: {key}')
        record_lengths[key] += int(count)
    for column in feature_columns:
        values = hours[column].to_numpy(dtype=np.float64)
        finite = np.isfinite(values)
        if np.any(~finite & ~np.isnan(values)):
            raise ValueError(f'Nonfinite observed value in {column}')
        feature = stats[column]
        feature['missing'] += int(np.isnan(values).sum())
        feature['observed'] += int(finite.sum())
        if finite.any():
            minimum, maximum = float(values[finite].min()), float(values[finite].max())
            feature['minimum'] = minimum if feature['minimum'] is None else min(feature['minimum'], minimum)
            feature['maximum'] = maximum if feature['maximum'] is None else max(feature['maximum'], maximum)
            if column in observed_values:
                observed_values[column].extend(values[finite].tolist())
    example_rows = hours.loc[
        (hours['hospital'] == example_key[0])
        & (hours['patient_id'] == example_key[1]),
        [*plot_columns, 'ICULOS', 'SepsisLabel'],
    ]
    trajectory.extend(example_rows.to_dict(orient='records'))

if hour_count != sum(record_lengths.values()) or any(length == 0 for length in record_lengths.values()):
    raise ValueError('Training Parquet rows do not match the selected training records')
lengths = list(record_lengths.values())
missingness = {
    column: 100 * value['missing'] / hour_count
    for column, value in stats.items()
}
print(f'Source-A training only: {len(training_records):,} records; {hour_count:,} hours')
print(f'Positive-label hours: {positive_hours:,} ({100 * positive_hours / hour_count:.2f}%)')
print(f'Example training record: {example_key[0]}:{example_key[1]}')
print('Raw values retained; no imputation or model training.')
print(f"{'Feature':<20} {'Missing %':>10} {'Observed':>10} {'Minimum':>14} {'Maximum':>14}")
for column in sorted(feature_columns, key=missingness.get, reverse=True):
    value = stats[column]
    print(f"{column:<20} {missingness[column]:>10.2f} {value['observed']:>10,} "
          f"{str(value['minimum']):>14} {str(value['maximum']):>14}")

## Understand the training data and cohort charts

The counts, missingness rates, and record lengths below are calculated from the local combined Parquet each time this notebook runs. Record counts are ICU stays, not necessarily unique people. Source A is split at the record level into approximately 55% training, 15% calibration, 15% validation, and 15% internal test using the project's `split_patients` function and seed 42. Source B is external test and is not included in the charts.

### 1. Hourly labels and record lengths

- The first chart shows counts of supplied zero and one labels for source-A training hours.
- The record-length histogram counts each training record once. Recorded hours are not necessarily complete ICU stays.
- The positive label is already shifted six hours early and persists afterward. Do not shift it again. Accuracy alone is misleading for this imbalanced outcome.

### 2. Feature missingness

- Bars show the percentage of missing training-hour values for each input used by this project.
- A longer bar means fewer measurements were available, not that values were more abnormal. Missingness alone does not establish corruption or justify dropping a feature.
- No missing values are filled in for this inspection.

### 3. Raw-value distributions

The four histograms show observed training values for HR (beats/minute), MAP (mmHg), Temp (degrees Celsius), and Resp (breaths/minute). The logarithmic count axis helps show tails; extreme values are candidates for review, not automatically errors. No values are clipped or removed.

### 4. Illustrative record trajectory

The final figure plots a deterministic source-A training stay. Dots show observed measurements; missing values remain gaps. The dashed line marks the first positive supplied label, not clinical onset or a model alert. This single record is not a representative cohort summary. Keep its patient-level chart local and clear notebook outputs before sharing or committing.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')
axes[0].bar(['Label 0', 'Label 1'], [hour_count - positive_hours, positive_hours])
axes[0].set(title='Training: supplied hourly labels', ylabel='Hourly observations')
axes[1].hist(lengths, bins=40, edgecolor='white')
axes[1].set(title='Training: full record-length range', xlabel='Recorded hours', ylabel='Records')
plt.show()

items = sorted(missingness.items(), key=lambda item: item[1])
fig, ax = plt.subplots(figsize=(10, 11), layout='constrained')
ax.barh([column for column, _ in items], [percent for _, percent in items])
ax.set(xlim=(0, 100), xlabel='Missing hourly values (%)', title='Training: original missingness')
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12, 8), layout='constrained')
for ax, column in zip(axes.flat, plot_columns, strict=True):
    values = observed_values[column]
    if values:
        ax.hist(values, bins=60, edgecolor='white')
        ax.set_yscale('log')
    else:
        ax.text(0.5, 0.5, 'No observed values', ha='center', transform=ax.transAxes)
    ax.set(title=f'{column}: observed values, full range', xlabel=column, ylabel='Count (log scale)')
fig.suptitle('Training: inspect tails and extremes; no clipping or filling')
plt.show()


## 5. One illustrative ICU-stay trajectory

The next figure has five vertically aligned panels: **HR**, **MAP**, **Temp**, **Resp**, and the **supplied sepsis label**. The measurements use the units listed above; the label is either 0 or 1.

- **Shared X-axis:** original `ICULOS`, the recorded ICU-hour index. The first available hour need not be 1.
- Dots mark observed hourly values. Lines connect adjacent observed hours; missing measurements remain gaps, not zeros or filled estimates.
- **Interpretation:** inspect sudden jumps, extreme values, sustained changes and long measurement gaps. None is automatically classified as an error.
- The bottom step plot shows the supplied label over time. The red dashed line marks its **first positive hour**, not clinical onset or a model alert. The label is a target, never an input feature.
- The notebook chooses the first sorted training record that is ever-positive but not positive in its first recorded row. If none exists, it uses the first training record. This selection avoids an ambiguous first-row-positive example when possible, but is not a representative sample of the cohort.
- Changes near the label boundary do not establish causation or prove early-warning performance. No model is trained or evaluated here.

**Keep patient-level trajectories local and clear outputs before committing or sharing.**


In [ ]:
icu_hours = [row['ICULOS'] for row in trajectory]
positive_times = [row['ICULOS'] for row in trajectory if row['SepsisLabel'] == 1]
fig, axes = plt.subplots(5, 1, figsize=(12, 10), sharex=True, layout='constrained')
units = ('beats/min', 'mmHg', 'degrees C', 'breaths/min')
for ax, column, unit in zip(axes[:4], plot_columns, units, strict=True):
    values = [float('nan') if row[column] is None else row[column] for row in trajectory]
    ax.plot(icu_hours, values, marker='.', linewidth=1)
    ax.set_ylabel(f'{column}\n({unit})')
    ax.grid(alpha=0.2)
axes[4].step(icu_hours, [row['SepsisLabel'] for row in trajectory], where='post')
axes[4].set(yticks=[0, 1], ylabel='Supplied label', xlabel='Original ICULOS')
if positive_times:
    for ax in axes:
        ax.axvline(positive_times[0], linestyle='--', color='red')
fig.suptitle('One illustrative training record: raw values, missingness remains gaps')
plt.show()
print('Dashed line: first positive supplied label, NOT clinical onset or a model alert.')
print('This deterministic example is not representative of the entire cohort.')
